# Jawad Hassan
# 2230-0035
# BS AI
# ANN
# Lab 11


### Task:
### LSTM WITH HAR

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler


# STEP 1: Load Dataset (Direct from UCI to get raw sequences)

print("Downloading and extracting raw HAR dataset...")
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip
!unzip -q "UCI HAR Dataset.zip"


# STEP 2: Preprocess Data (Sequence Creation)

# The raw dataset is split into text files for each axis. We must stack them into a 3D array.
def load_file(filepath):
    # Read space-separated values
    return pd.read_csv(filepath, header=None, delim_whitespace=True).values

def load_signals(subset):
    path = f'UCI HAR Dataset/{subset}/Inertial Signals/'
    # Using Total Accelerometer and Body Gyroscope (6 features per time step)
    files = [
        f'total_acc_x_{subset}.txt', f'total_acc_y_{subset}.txt', f'total_acc_z_{subset}.txt',
        f'body_gyro_x_{subset}.txt', f'body_gyro_y_{subset}.txt', f'body_gyro_z_{subset}.txt'
    ]
    signals = [load_file(path + f) for f in files]
    # Stack the 6 2D arrays into a single 3D array: (samples, 128 time_steps, 6 features)
    return np.dstack(signals)

X_train = load_signals('train')
X_test = load_signals('test')

# STEP 3: Preprocess Labels (Filtering for the 4 specific activities)

# The original dataset has 6 labels:
# 1:Walking, 2:Walking_Upstairs, 3:Walking_Downstairs, 4:Sitting, 5:Standing, 6:Laying
y_train_raw = load_file('UCI HAR Dataset/train/y_train.txt')
y_test_raw = load_file('UCI HAR Dataset/test/y_test.txt')

def map_to_four_classes(y):
    y_mapped = np.copy(y)
    # Group all 3 walking variations into a single "Walking" class (Index 0)
    y_mapped[(y == 1) | (y == 2) | (y == 3)] = 0
    # Map the remaining static postures
    y_mapped[y == 4] = 1 # Sitting
    y_mapped[y == 5] = 2 # Standing
    y_mapped[y == 6] = 3 # Laying
    return y_mapped

y_train = map_to_four_classes(y_train_raw)
y_test = map_to_four_classes(y_test_raw)

# Convert labels to one-hot encoded format for the neural network
y_train_cat = to_categorical(y_train, num_classes=4)
y_test_cat = to_categorical(y_test, num_classes=4)


# STEP 4: Preprocess Data (Normalization)

# LSTMs require 3D input, but standard scalers expect 2D data.
# We flatten the data, scale it, and reshape it back to 3D.
scaler = StandardScaler()
samples, timesteps, features = X_train.shape

# Fit and scale training data
X_train_reshaped = X_train.reshape(-1, features)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(samples, timesteps, features)

# Scale testing data (using the same scaler fitted on train data)
samples_test = X_test.shape[0]
X_test_reshaped = X_test.reshape(-1, features)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(samples_test, timesteps, features)


# STEP 5: Build Simple LSTM Model

model = Sequential()
# Input shape is (128 time steps, 6 features)
model.add(LSTM(64, input_shape=(timesteps, features)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
# Output layer uses softmax for multi-class classification (4 classes)
model.add(Dense(4, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


# STEP 6: Train Model

print("\nTraining the LSTM Model...")
history = model.fit(X_train, y_train_cat, epochs=15, batch_size=64, validation_split=0.2, verbose=1)


# STEP 7: Evaluate Model

loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n======================================")
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"======================================")


# STEP 8: Predict Activity Labels

predictions = model.predict(X_test)
# Convert probability arrays back to single class indices
predicted_classes = np.argmax(predictions, axis=1)

activity_dictionary = {0: 'Walking', 1: 'Sitting', 2: 'Standing', 3: 'Laying'}

print("\n--- Sample Predictions vs Actual ---")
# Show the first 10 predictions from the test set
for i in range(10):
    actual_label = activity_dictionary[y_test[i][0]]
    predicted_label = activity_dictionary[predicted_classes[i]]

    # Add a visual marker if the prediction is correct
    match = "✅" if actual_label == predicted_label else "❌"

    print(f"Sample {i+1:02d}: Actual = {actual_label:<10} | Predicted = {predicted_label:<10} {match}")

/tmp/ipykernel_8277/2786268382.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values
/tmp/ipykernel_8277/2786268382.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values
/tmp/ipykernel_8277/2786268382.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values
/tmp/ipykernel_8277/2786268382.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespa


Training the LSTM Model...
Epoch 1/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.7177 - loss: 0.6754 - val_accuracy: 0.8967 - val_loss: 0.3780
Epoch 2/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8699 - loss: 0.3793 - val_accuracy: 0.9014 - val_loss: 0.3796
Epoch 3/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9294 - loss: 0.2199 - val_accuracy: 0.8946 - val_loss: 0.3097
Epoch 4/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9221 - loss: 0.2515 - val_accuracy: 0.9048 - val_loss: 0.3292
Epoch 5/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9459 - loss: 0.1546 - val_accuracy: 0.9041 - val_loss: 0.3604
Epoch 6/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9554 - loss: 0.1342 - val_accuracy: 0.8960 - val_loss: 0.3644
Epoch 7/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9566 - loss: 0.1129 - val_accuracy: 0.8973 - val_loss: 0.3700
Epoch 8/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9546 - loss: 0.112